# U05 · 神经网络与反向传播实战

U4 你只会**回归**（输出一个连续值，比如 y=2x+3）。
U5 进入**分类**（输出离散标签，比如「猫/狗」「正面/负面」），翻译模型本质就是分类。

**本单元目标**：
1. 写多层 MLP（多层感知机）
2. 理解 Softmax + CrossEntropyLoss（**翻译模型的损失函数**）
3. 第一次用 DataLoader 做 mini-batch 训练
4. 看懂 train/val 曲线，识别过拟合


## §1 分类 vs 回归

| 任务 | 输出 | 损失函数 | 例子 |
|------|------|---------|------|
| 回归 | 连续值（一个数） | MSE | 房价预测、y=2x+3 |
| 二分类 | 两个类别 | BCELoss | 垃圾邮件、肿瘤良性/恶性 |
| 多分类 | N 个类别（**最常用**） | CrossEntropyLoss | 数字识别(10类)、词汇预测(几万类) |

**翻译模型 = 多分类**：每一步都从词表里选一个词，「选词」= 多分类。


## §2 Softmax：把分数变成概率

假设网络最后一层输出 3 个值（叫 **logits**，原始分数）：
```
z = [2.0, 1.0, 0.1]   ← 三类的「打分」，但不是概率
```

想变成概率（每类多大可能性）需要满足：
1. 每个值 ≥ 0
2. 加起来 = 1

**Softmax 公式**：
$$\text{softmax}(z_i) = \frac{e^{z_i}}{\sum_j e^{z_j}}$$

三步：
1. 全部 `exp`：保证非负
2. 求和：得到分母
3. 各自除以总和：归一化到 [0,1]，加起来 = 1


In [6]:
import torch
import torch.nn.functional as F

z = torch.tensor([2.0, 1.0, 0.1]) # (3,)

# 手动版
exp_z = torch.exp(z)            # [7.39, 2.72, 1.11]
p_manual = exp_z / exp_z.sum()  # [0.66, 0.24, 0.10]
print('手动 softmax:', p_manual)
print('和:', p_manual.sum())

# PyTorch 版
p = F.softmax(z, dim=-1)
print('PyTorch softmax:', p)


手动 softmax: tensor([0.6590, 0.2424, 0.0986])
和: tensor(1.0000)
PyTorch softmax: tensor([0.6590, 0.2424, 0.0986])


## §3 CrossEntropyLoss：分类的标准损失

假设真实标签是「类别 0」，模型预测概率是 `[0.66, 0.24, 0.10]`。

**直觉**：模型给真实类的概率越高越好。
**Cross-Entropy（交叉熵）公式**：
$$L = -\log(p_{\text{true class}})$$

- 模型给真实类预测 0.99 → `-log(0.99) = 0.01` 损失很小 ✅
- 模型给真实类预测 0.10 → `-log(0.10) = 2.30` 损失很大 ❌
- 模型给真实类预测 0.0001 → `-log(0.0001) = 9.21` 损失爆炸

**关键**：PyTorch 的 `nn.CrossEntropyLoss` **内部已经包含 Softmax**！
所以你的网络最后一层**不要**加 Softmax，直接输出 logits。


In [24]:
import torch
import torch.nn as nn

logits = torch.tensor([[2.0, 0.8, 0.1],     # 样本1
                       [0.8, 0.5, 0.8]])    # 样本2
labels = torch.tensor([0, 1])                # 样本1真值=类0, 样本2真值=类1

loss_fn = nn.CrossEntropyLoss()
loss = loss_fn(logits, labels)
# print('loss:', loss)
print('loss:', loss.item())

# 看一下两个常见错误：
# 错误1：手动加了 softmax
print(torch.softmax(logits, dim=1))

wrong = nn.CrossEntropyLoss(reduction='none')(torch.log(logits), labels)
# wrong = nn.CrossEntropyLoss()(torch.softmax(logits, dim=1), labels)
print('手动 softmax 后再算（错的）:', wrong)
# 错误2：labels 写成 one-hot —— PyTorch 要的是整数索引！


loss: 0.8401730060577393
tensor([[0.6893, 0.2076, 0.1031],
        [0.3649, 0.2703, 0.3649]])
手动 softmax 后再算（错的）: tensor([0.3716, 1.4351])


### CrossEntropyLoss 的输入约定（重要）

```python
loss = nn.CrossEntropyLoss()(logits, labels)
```

| 参数 | shape | 含义 |
|------|-------|------|
| logits | `(batch, num_classes)` | 网络原始输出，**不要 softmax** |
| labels | `(batch,)` | **整数**索引，不是 one-hot |

**翻译场景的延伸**：每一步预测下一个词，词表 5000 个词
- logits: `(batch, 5000)`
- labels: `(batch,)`，每个样本是 0~4999 的整数


## §4 多层 MLP（多层感知机）

**结构**：Linear → 激活 → Linear → 激活 → ... → Linear（最后输出层不加激活）

**为什么要多层？** U4 你已经知道：纯线性层叠多少层都还是线性，必须加非线性激活才有威力。

### 常见激活函数

| 函数 | 公式 | 用途 |
|------|------|------|
| ReLU | max(0, x) | **隐藏层默认选择**，简单快 |
| Sigmoid | 1/(1+e^-x) | 二分类输出层、门控（GRU 会用） |
| Tanh | (e^x-e^-x)/(e^x+e^-x) | 范围 [-1,1]，RNN 常用 |
| Softmax | 见上 | 多分类输出层（但 CE Loss 自带，不显式写） |


In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MLP(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, out_dim)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)        # 输出层不加激活！

model = MLP(in_dim=20, hidden_dim=64, out_dim=10)
print(model)
print('参数总数:', sum(p.numel() for p in model.parameters()))


MLP(
  (fc1): Linear(in_features=20, out_features=64, bias=True)
  (fc2): Linear(in_features=64, out_features=64, bias=True)
  (fc3): Linear(in_features=64, out_features=10, bias=True)
)
参数总数: 6154


## §5 数据加载：Dataset + DataLoader

U4 你把所有数据 `x`、`y` 一次喂进去训练。但是：
- 数据集大了内存装不下
- 全数据更新一次叫 batch GD，**慢且容易陷局部最优**
- 工业界都用 mini-batch（32 / 64 / 128 一组）

PyTorch 提供两个工具：
- `Dataset`：定义「我有什么数据，怎么取第 i 个」
- `DataLoader`：定义「怎么打包成 batch、要不要 shuffle、要几个进程」


In [4]:
import torch
from torch.utils.data import Dataset, DataLoader

class MyDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# 造点假数据
X = torch.randn(100, 20)        # 100 个样本，每个 20 维
y = torch.randint(0, 10, (100,))# 0~9 的标签

ds = MyDataset(X, y)
dl = DataLoader(ds, batch_size=8, shuffle=True)

# 看一个 batch 的形状
for xb, yb in dl:
    print('batch X:', xb.shape, 'batch y:', yb.shape)
    break
print('总 batch 数:', len(dl))


batch X: torch.Size([8, 20]) batch y: torch.Size([8])
总 batch 数: 13


## §6 完整训练模板（带 mini-batch）

U4 模板进化版：**外层循环 epoch，内层循环 batch**。

```python
for epoch in range(epochs):
    for xb, yb in train_loader:        # 内层：每个 batch
        optimizer.zero_grad()
        logits = model(xb)
        loss = loss_fn(logits, yb)
        loss.backward()
        optimizer.step()
```

**术语**：
- 1 个 batch 训练一次 = 1 个 step / iteration
- 所有 batch 训完一遍 = 1 个 epoch
- 100 个样本 / batch=10 → 1 epoch = 10 steps


## §7 过拟合 vs 欠拟合

训练集就像考试卷，模型在卷子上做得好（train loss 低），不代表它真懂（val loss 也低）。

**三种状态**：

| 状态 | train loss | val loss | 含义 |
|------|-----------|----------|------|
| 欠拟合 | 高 | 高 | 模型太弱，没学会 |
| **刚好** | 低 | 低 | ✅ 想要的状态 |
| 过拟合 | 很低 | 高 | 死记答案，不会迁移 |

**怎么发现过拟合**：训练时同时画 train_loss 和 val_loss 曲线
- 两条都下降 → 在学习
- train 继续降 / val 开始反弹 ↑ → 过拟合

**初步对策**（U6 详细学）：
- 加更多数据
- 减小模型容量（层数、隐藏维度）
- Dropout、L2 正则、早停


## 小结

| 概念 | 一句话记忆 |
|------|----------|
| 多分类 | 输出 N 个 logits |
| Softmax | logits → 概率（手算用，但训练时不写） |
| CrossEntropyLoss | 自带 Softmax，输入 logits + 整数标签 |
| MLP | Linear→ReLU→Linear→ReLU→Linear，最后一层不激活 |
| DataLoader | 自动打包 batch + shuffle |
| 训练循环 | epoch 套 batch，5 步循环不变 |
| 过拟合 | train 降 val 升，要早停 / 正则 |

做 `exercises.ipynb` 进入实战。
